# PF4 Heart Failure Associations from HERMES GWAS

This notebook processes heart failure summary statistics from the HERMES dataset to identify SNP associations within the PF4 genomic region.

In [1]:
import requests
import pandas as pd
import json
from pathlib import Path

pd.set_option("display.max_columns", None)

In [2]:
region_path = Path("../data/interim/ensembl/region.json")

hermes_path = Path("../data/raw/hermes/heart_failure.tsv")

out_dir = Path("../data/interim/hermes")
out_dir.mkdir(parents=True, exist_ok=True)

out_associations_csv = out_dir / "associations.csv"
out_summary_json = out_dir / "summary.json"

In [3]:
with open(region_path, "r") as f:
    pf4_region = json.load(f)

print("Gene:", pf4_region["gene_symbol"])
print("Assembly:", pf4_region["assembly_name"])
print(
    "PF4 region:",
    f"chr{pf4_region['chromosome']}:{pf4_region['region_start']}-{pf4_region['region_end']}"
)

Gene: PF4
Assembly: GRCh38
PF4 region: chr4:73930811-74032027


In [4]:
if not hermes_path.exists():
    raise FileNotFoundError(
        f"HERMES file not found at: {hermes_path}\n"
    )

In [5]:
hermes_preview = pd.read_csv(
    hermes_path,
    sep="\t",
    nrows=5
)

hermes_preview

,MarkerName,Allele1,Allele2,Freq1,FreqSE,MinFreq,MaxFreq,Effect,StdErr,P.value,Direction,HetISq,HetChiSq,HetDf,HetPVal,TotalSampleSize,nstudies,final_rsid,CHR,BP
0,1:100000012,t,g,0.2582,0.0208,0.2257,0.3068,0.0088,0.0099,0.3758,--+-?+-++-+-++++--+++-----++,0.0,17.928,26,0.8784,564738,27,rs10875231,1,100000012
1,1:100000827,t,c,0.3194,0.0295,0.2733,0.3910,0.0108,0.0094,0.2532,--+-?+++-++--++?+-+++-----++,0.0,20.478,25,0.7213,556506,26,rs6678176,1,100000827
2,1:100000843,t,c,0.9374,0.0098,0.9147,0.9538,-0.0150,0.0183,0.4124,-+++?---+---++++--?--++---+-,0.0,20.905,25,0.6979,559493,26,rs78286437,1,100000843
3,1:100000989:ID,d,i,0.9375,0.0131,0.9161,0.9588,-0.0296,0.0253,0.2416,-??----++?-???++--?--+?--?+?,13.8,19.712,17,0.2892,155499,18,rs146963890,1,100000989
4,1:100001138,a,g,0.9695,0.0076,0.9641,0.9880,0.0209,0.0334,0.5314,-????-+++??+-+++-??-+-?-+??+,0.0,13.664,16,0.6238,481945,17,rs144406489,1,100001138


In [6]:
# HERMES positions are treated as GRCh37 coordinates.
# Therefore, we retrieve the PF4 region from Ensembl GRCh37
# to avoid filtering a GRCh37 dataset with GRCh38 coordinates.

hermes_assembly = "GRCh37"
flanking_bp = int(pf4_region["flanking_bp"])

if hermes_assembly == pf4_region["assembly_name"]:
    hermes_region = {
        "gene_symbol": pf4_region["gene_symbol"],
        "chromosome": str(pf4_region["chromosome"]),
        "gene_start": int(pf4_region["gene_start"]),
        "gene_end": int(pf4_region["gene_end"]),
        "region_start": int(pf4_region["region_start"]),
        "region_end": int(pf4_region["region_end"]),
        "assembly_name": pf4_region["assembly_name"],
        "flanking_bp": flanking_bp,
    }

elif hermes_assembly == "GRCh37":
    server = "https://grch37.rest.ensembl.org"
    endpoint = f"/lookup/symbol/homo_sapiens/{pf4_region['gene_symbol']}"

    response = requests.get(
        server + endpoint,
        headers={
            "Content-Type": "application/json",
            "Accept": "application/json"
        },
        timeout=30
    )

    if not response.ok:
        response.raise_for_status()

    pf4_grch37 = response.json()

    hermes_region = {
        "gene_symbol": pf4_region["gene_symbol"],
        "chromosome": str(pf4_grch37["seq_region_name"]),
        "gene_start": int(pf4_grch37["start"]),
        "gene_end": int(pf4_grch37["end"]),
        "region_start": max(1, int(pf4_grch37["start"]) - flanking_bp),
        "region_end": int(pf4_grch37["end"]) + flanking_bp,
        "assembly_name": "GRCh37",
        "flanking_bp": flanking_bp,
    }

else:
    raise ValueError(f"Unsupported HERMES assembly: {hermes_assembly}")

print("HERMES filtering region:")
print(
    f"chr{hermes_region['chromosome']}:"
    f"{hermes_region['region_start']}-{hermes_region['region_end']}"
)
print("Assembly:", hermes_region["assembly_name"])

HERMES filtering region:
chr4:74796794-74897841
Assembly: GRCh37


In [7]:
usecols = [
    "MarkerName",
    "Allele1",
    "Allele2",
    "Freq1",
    "FreqSE",
    "MinFreq",
    "MaxFreq",
    "Effect",
    "StdErr",
    "P.value",
    "Direction",
    "HetISq",
    "HetChiSq",
    "HetDf",
    "HetPVal",
    "TotalSampleSize",
    "nstudies",
    "final_rsid",
    "CHR",
    "BP"
]

matched_chunks = []

region_chromosome = str(hermes_region["chromosome"]).replace("chr", "")
region_start = int(hermes_region["region_start"])
region_end = int(hermes_region["region_end"])

for chunk in pd.read_csv(
    hermes_path,
    sep="\t",
    usecols=usecols,
    chunksize=500_000
):
    chunk["CHR_filter"] = (
        chunk["CHR"]
        .astype(str)
        .str.replace("chr", "", regex=False)
    )

    chunk["BP_filter"] = pd.to_numeric(chunk["BP"], errors="coerce")

    matched = chunk[
        (chunk["CHR_filter"] == region_chromosome) &
        (chunk["BP_filter"].between(region_start, region_end))
    ].copy()

    matched = matched.drop(columns=["CHR_filter", "BP_filter"])

    if not matched.empty:
        matched_chunks.append(matched)

if matched_chunks:
    hermes_matches_df = pd.concat(matched_chunks, ignore_index=True)
else:
    hermes_matches_df = pd.DataFrame(columns=usecols)

hermes_matches_df.shape

(192, 20)

In [8]:
hermes_associations_df = hermes_matches_df.rename(columns={
    "final_rsid": "rsID",
    "MarkerName": "HERMES_marker_name",
    "CHR": "HERMES_CHR",
    "BP": "HERMES_BP",
    "Allele1": "HERMES_Allele1",
    "Allele2": "HERMES_Allele2",
    "Freq1": "HERMES_Allele1_freq",
    "FreqSE": "HERMES_freq_se",
    "MinFreq": "HERMES_min_freq",
    "MaxFreq": "HERMES_max_freq",
    "Effect": "HERMES_effect",
    "StdErr": "HERMES_std_error",
    "P.value": "HERMES_p_value",
    "Direction": "HERMES_direction",
    "HetISq": "HERMES_het_i_sq",
    "HetChiSq": "HERMES_het_chi_sq",
    "HetDf": "HERMES_het_df",
    "HetPVal": "HERMES_het_p_value",
    "TotalSampleSize": "HERMES_total_sample_size",
    "nstudies": "HERMES_n_studies"
})

hermes_associations_df.head()

,HERMES_marker_name,HERMES_Allele1,HERMES_Allele2,HERMES_Allele1_freq,HERMES_freq_se,HERMES_min_freq,HERMES_max_freq,HERMES_effect,HERMES_std_error,HERMES_p_value,HERMES_direction,HERMES_het_i_sq,HERMES_het_chi_sq,HERMES_het_df,HERMES_het_p_value,HERMES_total_sample_size,HERMES_n_studies,rsID,HERMES_CHR,HERMES_BP
0,4:74796915,a,g,0.1183,0.0149,0.0730,0.1360,0.0054,0.0136,0.6915,+-++----+----+++--?++-+-++--,4.1,27.117,26,0.4032,561591.0,27,rs164617,4,74796915
1,4:74797139,t,g,0.1188,0.0150,0.0730,0.1360,0.0101,0.0136,0.4593,+-++?---+----++?--+++-+-++--,12.2,28.485,25,0.2860,556510.0,26,rs352024,4,74797139
2,4:74797752,a,g,0.8804,0.0142,0.8640,0.9267,-0.0072,0.0135,0.5936,----?+++-++++---++?--+-+-+++,0.0,21.862,25,0.6437,559611.0,26,rs4694653,4,74797752
3,4:74797946,a,t,0.8815,0.0149,0.8640,0.9270,-0.0077,0.0134,0.5654,-+--++++-++++---++-----+--++,17.9,32.903,27,0.2003,566835.0,28,rs352023,4,74797946
4,4:74798010,a,g,0.0418,0.0109,0.0195,0.0654,-0.0001,0.0230,0.9958,-?--?+++--++-++-+-?++----+--,0.0,23.779,24,0.4743,557089.0,25,rs78267059,4,74798010


In [9]:
numeric_cols = [
    "HERMES_CHR",
    "HERMES_BP",
    "HERMES_Allele1_freq",
    "HERMES_freq_se",
    "HERMES_min_freq",
    "HERMES_max_freq",
    "HERMES_effect",
    "HERMES_std_error",
    "HERMES_p_value",
    "HERMES_het_i_sq",
    "HERMES_het_chi_sq",
    "HERMES_het_df",
    "HERMES_het_p_value",
    "HERMES_total_sample_size",
    "HERMES_n_studies"
]

for col in numeric_cols:
    hermes_associations_df[col] = pd.to_numeric(
        hermes_associations_df[col],
        errors="coerce"
    )

hermes_associations_df = hermes_associations_df.sort_values(
    "HERMES_p_value",
    ascending=True
)

hermes_associations_df.head()

,HERMES_marker_name,HERMES_Allele1,HERMES_Allele2,HERMES_Allele1_freq,HERMES_freq_se,HERMES_min_freq,HERMES_max_freq,HERMES_effect,HERMES_std_error,HERMES_p_value,HERMES_direction,HERMES_het_i_sq,HERMES_het_chi_sq,HERMES_het_df,HERMES_het_p_value,HERMES_total_sample_size,HERMES_n_studies,rsID,HERMES_CHR,HERMES_BP
161,4:74869496,a,g,0.0262,0.0104,0.0144,0.0558,0.0765,0.0361,0.03391,-????+??-?+-+-+--+?+?+-++??+,25.2,21.402,16,0.1636,505250.0,17,rs111720070,4,74869496
63,4:74824817,t,g,0.9838,0.0032,0.9806,0.9899,-0.1117,0.0627,0.07492,-????-+-+??+--++--?-?+??????,4.8,13.651,13,0.3988,106285.0,14,rs139071863,4,74824817
149,4:74860140,a,g,0.0182,0.0037,0.0108,0.0244,0.0964,0.0559,0.08499,+????+?+-??-++--+??+?-??+-??,0.0,11.345,13,0.5819,108549.0,14,rs17813879,4,74860140
190,4:74895277,t,c,0.0186,0.0060,0.0104,0.0265,0.0653,0.0428,0.12730,+????+?-???-++--+??+?+??+-?+,0.0,10.380,13,0.6626,473216.0,14,rs17814280,4,74895277
168,4:74877144,a,g,0.3609,0.0230,0.2940,0.4325,-0.0146,0.0107,0.17330,-?+??--++?-++++--??-?-+-+??-,0.0,17.488,18,0.4898,510273.0,19,rs2668076,4,74877144


In [10]:
rows_before_deduplication = len(hermes_associations_df)

hermes_associations_df = hermes_associations_df.drop_duplicates()

rows_after_deduplication = len(hermes_associations_df)
duplicate_rows_removed = rows_before_deduplication - rows_after_deduplication

duplicate_rows_removed

0

In [11]:
hermes_associations_df.to_csv(out_associations_csv, index=False)

print("Saved:", out_associations_csv)

Saved: ../data/interim/hermes/associations.csv


In [12]:
summary = {
    "source_dataset": "HERMES Heart Failure summary statistics without UK Biobank samples",
    "phenotype": "Heart failure",
    "region": (
        f"chr{hermes_region['chromosome']}:"
        f"{hermes_region['region_start']}-{hermes_region['region_end']}"
    ),
    "region_assembly": hermes_region["assembly_name"],
    "associations": int(len(hermes_associations_df)),
    "unique_rsIDs": int(hermes_associations_df["rsID"].nunique()),
}

with open(out_summary_json, "w") as f:
    json.dump(summary, f, indent=4)

summary

{'source_dataset': 'HERMES Heart Failure summary statistics without UK Biobank samples',
 'phenotype': 'Heart failure',
 'region': 'chr4:74796794-74897841',
 'region_assembly': 'GRCh37',
 'associations': 192,
 'unique_rsIDs': 192}